# Automatisation de la catégorisation des produits d'une marketplace et extraction de leurs attributs  - Nricher

### Import du fichier

In [1]:
import pandas as pd

# Provide the path to your .xlsb file
file_path = r'20210614 Ecommerce sales.xlsb'

# List available sheet names
with pd.ExcelFile(file_path, engine='pyxlsb') as xlsb:
    print("Available sheets:", xlsb.sheet_names)

# Read the first sheet 
df = pd.read_excel(file_path, engine='pyxlsb', sheet_name=0)
df.head()

Available sheets: ['20210614 Ecommerce sales']


,Cod_cmd,Libellé produit,Vendeur,Univers,Nature,Date de commande,Montant cmd,Quantité,Prix transport,Délai transport annoncé
0,182210782,Table basse carrée detroit design industriel,Autre vendeur,Canapé Salon Séjour,Table basse,44216,244,4,6.67,10.0
1,182082437,Ours en peluche géant 150 cm brun,Autre vendeur,Enfant Bébé,Peluche,44213,28,1,9.92,10.0
2,182095765,Ours en peluche géant 100 cm blanc,Autre vendeur,Enfant Bébé,Peluche,44214,15,1,9.92,10.0
3,182615392,Lot de 4 chaises mia noires pour salle à manger,Autre vendeur,Canapé Salon Séjour,Chaise,44219,385,2,20.75,10.0
4,184222081,Meuble tv falko bois blanc et gris,Autre vendeur,Canapé Salon Séjour,Meuble tv,44238,61,1,19.08,10.0


# I. Correction et catégorisation automatique des produits

### 1. TF-IDF et Régression Logistique

In [ ]:
# 2. Import des libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

In [ ]:
# Features and target
X = df['Libellé produit']
y = df['Nature']

# Supprimer les classes rares (qui apparaissent une seule fois)
value_counts = df['Nature'].value_counts()
valid_classes = value_counts[value_counts > 1].index
df = df[df['Nature'].isin(valid_classes)].reset_index(drop=True)

# Forcer la colonne 'Libellé produit' à être des strings
df['Libellé produit'] = df['Libellé produit'].astype(str)

# Encode target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# Vectorize text
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Using a simple Logistic Regression first
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Evaluate the model 

from sklearn.metrics import accuracy_score

# Predictions
y_pred = model.predict(X_test_vec)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
# print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Save the model if good enough
import joblib
joblib.dump(model, 'category_classifier.joblib')
joblib.dump(vectorizer, 'tfidf_vectorizer.joblib')


## 2. Fine-tuning du CamemBERT

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import CamembertTokenizer, CamembertForSequenceClassification
import joblib

In [ ]:
X = df['Libellé produit'].astype(str)
y = df['Nature']

# 2. Filtrer les classes rares
value_counts = y.value_counts()
valid_classes = value_counts[value_counts > 1].index
df = df[df['Nature'].isin(valid_classes)].reset_index(drop=True)
X = df['Libellé produit'].astype(str)
y = df['Nature']

# 3. Encoder les labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 4. Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# 5. Dataset PyTorch
class ProductDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts.iloc[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 6. Charger CamemBERT
model_name = 'camembert-base'
tokenizer = CamembertTokenizer.from_pretrained(model_name)
model = CamembertForSequenceClassification.from_pretrained(model_name, num_labels=len(le.classes_))

# 7. DataLoaders
train_dataset = ProductDataset(X_train, y_train, tokenizer)
test_dataset = ProductDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

# 8. Entraînement (3 epochs)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {avg_loss:.4f}")

# 9. Evaluation
model.eval()
preds = []
true_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)

        preds.extend(predictions.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(true_labels, preds)
print(f"Accuracy on test set: {accuracy:.4f}")

# 10. Sauvegarder model, tokenizer et label encoder
model.save_pretrained("camembert_category_classifier")
tokenizer.save_pretrained("camembert_category_classifier")
joblib.dump(le, 'label_encoder.joblib')

print("Modèle, tokenizer et label encoder sauvegardés.")

# 11. Détecter les mauvaises catégorisations
full_dataset = ProductDataset(X, y_encoded, tokenizer)
full_loader = DataLoader(full_dataset, batch_size=16)

model.eval()
all_preds = []

with torch.no_grad():
    for batch in full_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
        all_preds.extend(predictions.cpu().numpy())

# Comparer
df['Nature_predite'] = le.inverse_transform(all_preds)
df['Nature_reelle'] = le.inverse_transform(y_encoded)

# Garde uniquement les erreurs
df_erreurs = df[df['Nature_predite'] != df['Nature_reelle']][['Libellé produit', 'Nature_reelle', 'Nature_predite']]

# Sauvegarder
df_erreurs.to_csv('produits_mal_categorises.csv', index=False)

print(f"Nombre de produits mal étiquetés: {len(df_erreurs)}")
print("CSV sauvegardé: produits_mal_categorises.csv")


# II. Extraction des attributs

### 1. Extraction des dimensions

In [3]:
import re

# Patterns robustes et priorisés
patterns = [
    # 1. Dimensions combinées (90x190 cm, 110x40x45 cm)
    r'(\d+(?:[,.]\d+)?\s*[x×]\s*\d+(?:[,.]\d+)?(?:\s*[x×]\s*\d+(?:[,.]\d+)?)?\s*(?:cm|m|mm|pouces|po|inch|"|in))',
    
    # 2. Dimensions avec lettres (L150 x H200 cm)
    r'((?:[LlHhPpWwDdØø]\s*)?\d+(?:[,.]\d+)?\s*[x×]\s*(?:[LlHhPpWwDdØø]\s*)?\d+(?:[,.]\d+)?(?:\s*[x×]\s*(?:[LlHhPpWwDdØø]\s*)?\d+(?:[,.]\d+)?)?\s*(?:cm|m|mm|pouces|po|inch|"|in))',
    
    # 3. Diamètre (diamètre 50 cm, Ø30 cm)
    r'((?:diam(?:ètre|etre)?\.?|Ø|ø)\s*\d+(?:[,.]\d+)?\s*(?:cm|m|mm|pouces|po|inch|"|in))',
    
    # 4. Dimension simple AVEC unité (150 cm, 2 m, etc.)
    r'(\b\d+(?:[,.]\d+)?\s*(?:cm|m|mm|pouces|po|inch|"|in)\b)',
]

# Compile avec ignorecase
compiled_patterns = [re.compile(pattern, flags=re.IGNORECASE) for pattern in patterns]

def extract_dimensions(text):
    if pd.isna(text):
        return None
    text = str(text)

    for pattern in compiled_patterns:
        match = pattern.search(text)
        if match:
            dimension = match.group(1)
            # Remplace virgule par point pour cohérence (ex: "1,5 cm" => "1.5 cm")
            dimension = dimension.replace(',', '.').strip()
            return dimension
    return None


# Application sur ta colonne "Libellé produit"
df["Dimensions"] = df["Libellé produit"].apply(extract_dimensions)

# Résultat
print(df[['Libellé produit', 'Dimensions']].head())



                                   Libellé produit Dimensions
0     Table basse carrée detroit design industriel       None
1                Ours en peluche géant 150 cm brun     150 cm
2               Ours en peluche géant 100 cm blanc     100 cm
3  Lot de 4 chaises mia noires pour salle à manger       None
4               Meuble tv falko bois blanc et gris       None


### 2. Extraction des couleurs 

In [4]:
import unicodedata
from functools import lru_cache

# 1. Liste de couleurs enrichie
colors = [
    # Français
    'blanc', 'blanche', 'noir', 'noire', 'gris', 'grise', 'gris clair', 'gris foncé', 'bleu', 'bleue', 'bleu clair', 'bleu foncé', 'bleu marine',
    'rouge', 'vert', 'verte', 'vert clair', 'vert foncé', 'jaune', 'rose', 'rose clair', 'rose foncé',
    'marron', 'chocolat', 'taupe', 'beige', 'bordeaux', 'violet', 'violette', 'violet clair', 'violet foncé',
    'orange', 'dore', 'doree', 'argente', 'argentee', 'ecru', 'kaki', 'camel', 'ivoire', 'anthracite', 'saumon', 'turquoise', 'olive', 'cuivre', 'corail', 'ocre', 'lavande', 'multicolore',
    # Espagnol
    'blanco', 'blanca', 'negro', 'negra', 'gris', 'gris claro', 'gris oscuro', 'azul', 'azul claro', 'azul oscuro', 'azul marino',
    'rojo', 'roja', 'verde', 'verde claro', 'verde oscuro', 'amarillo', 'amarilla', 'rosa', 'rosa claro', 'rosa oscuro',
    'marron', 'chocolate', 'beige', 'burdeos', 'violeta', 'violeta claro', 'violeta oscuro',
    'naranja', 'dorado', 'dorada', 'plateado', 'plateada', 'crudo', 'cruda', 'caqui', 'camel', 'marfil', 'antracita', 'salmon', 'turquesa', 'oliva', 'cobre', 'coral', 'ocre', 'lavanda', 'multicolor',
    # Anglais
    'white', 'black', 'grey', 'gray', 'light grey', 'dark grey', 'light gray', 'dark gray',
    'blue', 'light blue', 'dark blue', 'navy blue',
    'red', 'green', 'light green', 'dark green', 'yellow', 'pink', 'light pink', 'dark pink',
    'brown', 'chocolate', 'beige', 'burgundy', 'purple', 'light purple', 'dark purple',
    'orange', 'gold', 'silver', 'ivory', 'khaki', 'camel', 'anthracite', 'salmon', 'turquoise', 'olive', 'copper', 'coral', 'ochre', 'lavender', 'multicolor'
]

# 2. Normalisation texte
def normalize_text(text):
    if not text:
        return ""
    text = str(text).lower()
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')
    return text

# 3. Trie couleurs par longueur décroissante
colors_sorted = sorted(colors, key=lambda x: -len(x))

# 4. Compile regex couleurs
colors_pattern = re.compile(r'\b(' + '|'.join(re.escape(color) for color in colors_sorted) + r')\b', flags=re.IGNORECASE)

# 5. Fonction extraction
@lru_cache(maxsize=None)
def extract_colors(text):
    text = normalize_text(text)
    matches = colors_pattern.findall(text)
    if not matches:
        return ['None']
    # Enlever doublons tout en gardant l'ordre
    seen = set()
    unique_matches = []
    for match in matches:
        match_lower = match.lower()
        if match_lower not in seen:
            seen.add(match_lower)
            unique_matches.append(match_lower)
    return unique_matches


# Application de la fonction
df['Couleurs'] = df['Libellé produit'].astype(str).map(extract_colors)


### Sauvegarde du nouveau fichier avec les attributs extraits

In [5]:
# 7. Sauvegarde dans un nouveau fichier Excel
df[['Libellé produit', 'Couleurs', 'Dimensions']].to_excel('ecom-extractedFeatures.xlsx', index=False)